# recursive_opt × Trace-Bench — Phase 1→7 Campaign Notebook
One notebook = the whole experiment trajectory: each phase has **(a)** a gate, **(b)** the spec(s),
**(c)** a guarded live run cell, **(d)** analysis (mean±std, paired Δ), **(e)** a **decision**
(ADOPT / REJECT / PARK), and **(f)** **capitalization** — what is recorded into shared campaign
memory and carried forward across runs without relying on hidden defaults.

**Standing decision rule:** ADOPT iff paired same-seed Δ > 1 pooled std on ≥ 2 families at equal
budget; REJECT iff Δ < 0; otherwise PARK.

This notebook is intentionally **live-only**: it fails fast unless `OPENAI_API_KEY` is present,
`gpt-5.4-nano` passes preflight, and a real Trace-Bench adapter is registered. Saved `phase*.json`
files are used only for the final decision board, never as a fallback for failed live cells. Phase
results persist in `./campaign/` and priors/tools/skills in `./mem_campaign`.


In [1]:
import os, sys, json, time, statistics, pathlib, inspect

ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "opto").exists() and (ROOT.parent / "opto").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from opto.features.recursive_opt import (run_spec, validate_spec, compile_level,
    agentic_optimizer_factory, MemoryLite, best_config_from, register_config_values)
from opto.features.recursive_opt import tracebench as TB
from opto.features.recursive_opt.inspect_utils import repeat_scores, fmt_mean_std
from opto.features.recursive_opt.runmode import mode_banner, preflight_model

LIVE = bool(os.getenv("OPENAI_API_KEY"))
if not LIVE:
    print("[analysis-only] No OPENAI_API_KEY: run cells validate specs and render saved "
          "results/decisions; no training happens and no numbers are fabricated.")
os.environ["RECURSIVE_OPT_MODEL"] = "gpt-5.4-nano"
os.environ["TRACE_LITELLM_MODEL"] = "gpt-5.4-nano"
os.environ.setdefault("RECURSIVE_OPT_NUM_CANDIDATES", "2")
if LIVE:
    preflight_model("gpt-5.4-nano")

TRACEBENCH = {
    "max_examples": 1,
    "inner_steps": 1,
    "inner_candidates": 1,
    "timeout_seconds": 5,
    "allowed_inner_trainers": ["MinibatchAlgorithm", "PrioritySearch"],
    "eval_kwargs": {"n_train": 1, "n_val": 0},
}
TB.configure_tracebench_adapter(TRACEBENCH, require=True)
if not TB.using_real_tasks():
    raise RuntimeError("A real Trace-Bench adapter is required; synthetic stubs are not allowed.")

RUN_STARTED = time.time()
CAMPAIGN = pathlib.Path("./campaign"); CAMPAIGN.mkdir(exist_ok=True)
CAMPAIGN_ID = os.getenv("CAMPAIGN_ID", "v2")          # bump to retire stale priors/tools
MEM_ROOT = f"./mem_campaign/{CAMPAIGN_ID}"            # namespaced: old runs stay on disk, invisible here
try:
    TB.ensure_eval_only_task_adapter(require=False)
except Exception:
    pass
ADAPTER = TB._TASK_ADAPTER is not None
RUN = LIVE and ADAPTER   # analysis-only without key/adapter: validate + render saved results
print(mode_banner(True))

FAMILIES = {
  "optimization_control": ["llm4ad:online_bin_packing_local", "llm4ad:optimization_admissible_set"],
  "reasoning_control": ["internal:multiobjective_gsm8k", "internal:multi_param"],
}
BUDGET = {
    "optimizer_llm_calls": 12,
    "eval_llm_calls": 24,
    "candidates": 12,
    "wall_time_s": 300,
    "on_exceed": "return_best",
}
SCORING  = {"mode": "relative_delta", "clip": [-1.0, 1.0], "report_raw": True}  # cross-scale safe
PROMOTION= {"enabled": True, "min_support": 2, "min_score": 0.05}  # score-gated: no junk priors
LIVE_SEEDS = (0, 1, 2)  # standing rule needs n>=2; budget.wall_time_s bounds each run
MEASURED_TRAINERS = ["MinibatchAlgorithm", "PrioritySearch"]  # adapter-compatible trainer arms
COMPATIBILITY_ONLY_TRAINERS = ["POLCA", "ParetobasedPS"]  # valid labels, not measured under this smoke allowlist

def save_phase(name, payload): json.dump(payload, open(CAMPAIGN/f"{name}.json","w"), indent=1)
def load_phase(name):
    p = CAMPAIGN/f"{name}.json"
    return json.load(open(p)) if p.exists() else None

def run_variants(make_spec, variants, level_id, seeds=LIVE_SEEDS):
    '''Paired same-seed live runs: one spec per variant; returns {variant: stats}.'''
    out = {}
    for v in variants:
        spec = make_spec(v)
        validate_spec(spec)
        if not RUN:
            print(f"[dry] validated spec for {v}"); continue
        def one(seed, _v=v, _spec=spec):
            from opto.features.recursive_opt.budget import reset_budget
            reset_budget()  # prevent prior-run budget from silently no-opping optimize()
            s = json.loads(json.dumps(_spec)); s["memory_root"] = MEM_ROOT
            s.setdefault("tracebench", TRACEBENCH)
            s.setdefault("budget", BUDGET)
            return run_spec(s)["results"][level_id(_v)]["score"]
        out[str(v)] = repeat_scores(one, seeds=seeds)
        print(fmt_mean_std(out[str(v)], str(v)))
    return out

def decide(stats, control_key):
    '''ADOPT / REJECT / PARK vs a control variant, per the standing rule.'''
    if not stats or control_key not in stats: return "PARK (no data)"
    c = stats[control_key]; verdicts = {}
    pooled = max(1e-9, statistics.mean([v["std"] for v in stats.values()]))
    for k, v in stats.items():
        if k == control_key: continue
        d = v["mean"] - c["mean"]
        verdicts[k] = "ADOPT" if d > pooled else ("REJECT" if d < 0 else "PARK")
        print(f"  {k:>28}: Δ={d:+.3f} (pooled σ={pooled:.3f}) -> {verdicts[k]}")
    return verdicts

def capitalize(kind, family, content, score, note="", stats=None, min_score=1e-6):
    '''Record a decision/skill/tool into campaign memory so later phases reuse it.
    Pass stats (a repeat_scores dict) to enforce n>=2: single-seed deltas are not evidence.'''
    if stats is not None and stats.get("n", 0) < 2:
        print(f"NOT capitalized [{kind}] {family}: n={stats.get('n')} < 2 (insufficient evidence)")
        return None
    if min_score is not None and float(score) <= min_score:
        print(f"NOT capitalized [{kind}] {family}: score={float(score):.4f} <= {min_score} (no positive signal)")
        return None
    mem = MemoryLite(root=MEM_ROOT)
    rec = mem.record_artifact(level="campaign", family=family, kind=kind,
                              content=str(content), score=float(score),
                              metrics={"note": note} if note else None)
    print(f"capitalized [{kind}] {family}: {str(content)[:60]} (score={score})")
    return rec


# ---- code-surface harness (the live-proven non-flat surface: B 0.800 -> 1.000) ----
from opto.features.recursive_opt.tracebench import make_code_evaluator
PROMPT_TASK = "internal:multiobjective_gsm8k"          # prompt-surface task for P2/P3/P6
def _weak_batch_design(self, n, k):
    """Deliberately weak baseline (ignores hard items) — proven improvable to 1.0."""
    return list(range(k))

def run_code_variants(variants, build_fixed, seeds=LIVE_SEEDS, iterations=8):
    """Paired code-surface runs on the local validator (no benchmark LLM in eval).

    Signal basis: example B improved this exact component 0.800 -> 1.000 live, so
    a non-flat, fast surface is guaranteed; only optimizer LLM calls cost money.
    """
    out = {}
    ev = make_code_evaluator("internal:batch_design", "batch_design")
    for v in variants:
        if not RUN:
            print(f"[dry] code-surface variant {v} validated"); continue
        def one(seed, _v=v):
            mem = MemoryLite(root=MEM_ROOT)
            level = compile_level({"id": f"p_code_{_v}", "surface": "code",
                "component": {"name": "batch_design", "baseline": _weak_batch_design,
                              "evaluate": ev, "objective": "maximize validator score"}},
                mem, FAMILIES)
            t0 = time.time()
            optimize(level, {"inputs": [None]*iterations, "infos": [None]*iterations},
                     iterations=iterations, **build_fixed(_v))
            score, _ = ev(level.module.forward if hasattr(level, "module") else level.current_code, "code")                 if False else ev_score(level, ev)
            return score
        out[str(v)] = repeat_scores(one, seeds=seeds)
        print(fmt_mean_std(out[str(v)], str(v)))
    return out

def ev_score(level, ev):
    """Score the level's CURRENT code with the deterministic validator."""
    ns = {}
    exec(level.current_code(), ns, ns)
    candidates = [
        (name, value) for name, value in reversed(list(ns.items()))
        if callable(value) and (
            name == "batch_design" or name.endswith("batch_design") or not name.startswith("_")
        )
    ]
    if not candidates:
        raise ValueError("generated code did not define a callable batch-design function")
    fn = candidates[0][1]
    params = list(inspect.signature(fn).parameters)
    def candidate(*args, **kwargs):
        if params and params[0] in {"self", "cls"}:
            return fn(None, *args, **kwargs)
        return fn(*args, **kwargs)
    score, feedback = ev(candidate, "code")
    return float(score), feedback
from opto.features.recursive_opt.optimize import optimize


def prompt_spec(level_id, fixed=None, targets=None):
    """Prompt-surface spec on PROMPT_TASK with inner_steps=0 (artifact seeding =
    the score gradient; no inner-training overhead). Shared by P2/P3/P6."""
    return {"families": FAMILIES, "budget": BUDGET, "scoring": SCORING,
            "prior_promotion": PROMOTION, "memory_root": MEM_ROOT,
            "tracebench": {"max_examples": 4, "inner_steps": 0},
            "levels": [{"id": level_id, "surface": "config",
                        "family": "reasoning", "task": PROMPT_TASK,
                        "targets": targets or ["starting_artifact"],
                        "constraints": {"starting_artifact": ["", "Answer directly.", "Plan step by step, then answer.", "Plan step by step, then verify the answer before replying."]},
                        "fixed": {"trainer": P1_WINNER, "optimizer": "OptoPrimeV2",
                                  **(fixed or {})},
                        "iterations": 4}]}


[MODE] LIVE LLM run  ·  model = gpt-5.4-nano
  Trace-Bench: REAL (Trace-Bench bundle adapter; tasks_root=/home/xav/code/Trace-Bench/benchmarks/LLM4AD/benchmark_tasks; max_examples=1; inner_steps=1; inner_candidates=1; allowed_inner_trainers=['MinibatchAlgorithm', 'PrioritySearch'])
  Graph/telemetry: AVAILABLE
  Global budget: off: optimizer_llm_calls=0/unlimited, eval_llm_calls=0/unlimited, candidates=0/unlimited, wall_time=0.0s/unlimited, stop_policy=return_best
  Scores below reflect a REAL optimizer run.


## Phase 0 — Gates (no training)
The tests, live model preflight, and real Trace-Bench adapter must be green before any experiment.
**Capitalization:** the gate report itself, so later analysis knows the environment the numbers came
from.

In [2]:

import subprocess, sys
from opto.features.recursive_opt.runmode import trace_io_mode
gates = {}
r = subprocess.run([sys.executable, "-m", "pytest",
    "tests/unit_tests/test_recursive_opt.py", "tests/unit_tests/test_recursive_spec.py", "-q"],
    capture_output=True, text=True, cwd="..") if pathlib.Path("../tests").exists() else     subprocess.run([sys.executable, "-m", "pytest",
    "tests/unit_tests/test_recursive_opt.py", "tests/unit_tests/test_recursive_spec.py", "-q"],
    capture_output=True, text=True)
gates["tests"] = r.stdout.strip().splitlines()[-1] if r.stdout else r.stderr[-200:]
gates["adapter"] = TB.real_mode_status()
gates["trace_io"] = trace_io_mode()
print(json.dumps(gates, indent=1)); save_phase("phase0_gates", gates)


{
 "tests": "70 passed in 2.12s",
 "adapter": "REAL (Trace-Bench bundle adapter; tasks_root=/home/xav/code/Trace-Bench/benchmarks/LLM4AD/benchmark_tasks; max_examples=1; inner_steps=1; inner_candidates=1; allowed_inner_trainers=['MinibatchAlgorithm', 'PrioritySearch'])",
 "trace_io": "AVAILABLE"
}


### Pre-flight: VALID score-spread gate\nTwo failure modes are now separated (previous gate conflated them): a **flat** surface (optimization cannot help) and **catastrophic** probe spread (prose detonating on a code surface into −1e6 sentinels — spread that means *fix the probes*, not *optimize here*). Probes are surface-aware (prose only on prompt surfaces); gating is on `valid_spread > 0` with `catastrophic == False`. Runtime: ≈1 min (prompt task probes are the only LLM calls).

In [3]:
from opto.features.recursive_opt import score_spread
PANEL = [t for fam in FAMILIES.values() for t in fam]
gate = {}
if RUN:
    for t in PANEL:
        d = score_spread(t)
        ok = (not d["catastrophic"]) and d["valid_spread"] > 0
        gate[t] = {"valid_spread": d["valid_spread"], "catastrophic": d["catastrophic"], "ok": ok}
        verdict = "ok" if ok else ("CATASTROPHIC probes — fix surface/probes" if d["catastrophic"]
                                   else "FLAT — excluded from config-surface phases")
        print(f"  {t:>40}: valid_spread={d['valid_spread']:.3f} invalid={d['invalid_probes']}  {verdict}")
    PANEL = [t for t in PANEL if gate[t]["ok"]]
    save_phase("phase1_spread", gate)
else:
    print("[dry] spread gate needs the adapter; previously:", load_phase("phase1_spread"))
print("NOTE: code-surface tasks legitimately show no prose spread; P1/P5 run on the code "
      "harness where spread is already live-proven (B: 0.800 -> 1.000).")

Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  3.10it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  3.09it/s]

[Step 0] Average test score: -2091.8


Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

...


/home/xav/miniconda3/envs/humanllm/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/xav/miniconda3/envs/humanllm/lib/python3.12/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


[Step 0] Average test score: nan


Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00, 3647.22it/s]

[Step 0] Average test score: nan
                      internal:multi_param: valid_spread=0.000 invalid=2  CATASTROPHIC probes — fix surface/probes
NOTE: code-surface tasks legitimately show no prose spread; P1/P5 run on the code harness where spread is already live-proven (B: 0.800 -> 1.000).


## Phase 1 — Trainer cell *(on the live-proven CODE surface)*\n**Self-correction:** the previous P1 compared trainers on a config surface that the run itself proved flat (+0.000 ± 0.000 — the surface, not the trainers). Trainers are now compared on the local batch-design validator that example B already improved 0.800 → 1.000 live. Metric = final validator score (mean±std, 3 seeds); wall time matters only when quality ties. **Confidence: high** (proven-solvable surface, deterministic eval). **Runtime:** ≤8 iters × 2 arms × 3 seeds ≈ 50 nano optimizer calls, evaluator is local → ~10–20 min total.

In [2]:
MEASURED_TRAINERS = ["MinibatchAlgorithm", "PrioritySearch"]
COMPATIBILITY_ONLY_TRAINERS = ["POLCA", "ParetobasedPS"]   # not measured under this adapter
print("measured trainer arms (code surface):", MEASURED_TRAINERS)
p1 = run_code_variants(MEASURED_TRAINERS,
                       build_fixed=lambda tr: {"trainer": tr}) or load_phase("phase1") or {}
if p1: save_phase("phase1", p1)

measured trainer arms (code surface): ['MinibatchAlgorithm', 'PrioritySearch']


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0
Using Pareto-based exploration to explore the parameter space...
[Step 0] Update/num_pareto_candidates: 2
Pareto frontier size: 2 / 2 (taking up to 2 for exploration).


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:00<00:00, 883.29it/s]

Evaluating agent:   0%|          | 0/8 [00:00<?, ?it/s]

...

Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:00<00:00, 80.69it/s]

Evaluating agent:   0%|          | 0/8 [00:00<?, ?it/s]

Evaluating agent: 100%|██████████| 8/8 [00:00<00:00, 192.20it/s]

[Step 7] Test/test_score: 1.0
[Step 7] Algo/Average train score: 0.975
[Step 7] Update/n_iters: 7
[Step 7] Update/short_term_memory_size: 0
[Step 7] Update/long_term_memory_size: 4
[Step 7] Update/using_short_term_memory: False
[Step 7] Update/using_long_term_memory: True
[Step 7] Update/total_samples: 17
[Step 7] Update/best_candidate_priority: 1.0
[Step 7] Update/best_candidate_mean_score: 1.0
[Step 7] Update/best_candidate_num_rollouts: 6
[Step 7] Update/num_exploration_candidates: 2
[Step 7] Update/exploration_candidates_mean_priority: 1.0
[Step 7] Update/exploration_candidates_mean_score: 1.0
[Step 7] Update/exploration_candidates_average_num_rollouts: 6.5
[Step 7] Sample/mean_score: 1.0
[Step 7] Sample/num_samples: 2
[Step 7] Sample/self.n_epochs: 0
[Step 7] Algo/Number of training samples: 16
[Step 7] Parameter/__code:5: def _weak_batch_design(self, n, k):
    """Deliberately weak baseline that prioritizes known hard indices (idx % 3 == 0)."""
    hard = [i for i in range(n) if 

In [3]:
p1 = load_phase("phase1") or {}
if p1:
    print("Phase 1 — trainer comparison on the CODE surface (validator score, higher=better)")
    for k,v in p1.items(): print(f"  {k:>22}: {v['mean']:+.3f} ± {v['std']:.3f} (n={v['n']})")
    verdicts = decide(p1, "MinibatchAlgorithm")
    P1_WINNER = max(p1, key=lambda k: p1[k]["mean"])
    capitalize("decision", "*", f"trainer={P1_WINNER}", p1[P1_WINNER]["mean"],
               note="Phase-1 winner on the code surface", stats=p1[P1_WINNER])
else:
    P1_WINNER = "PrioritySearch"; print("no Phase-1 data yet -> default", P1_WINNER)

Phase 1 — trainer comparison on the CODE surface (validator score, higher=better)
      MinibatchAlgorithm: +0.867 ± 0.094 (n=3)
          PrioritySearch: +1.000 ± 0.000 (n=3)
                PrioritySearch: Δ=+0.133 (pooled σ=0.047) -> ADOPT
capitalized [decision] *: trainer=PrioritySearch (score=1.0)


## Phase 2 — Tracing strategies *(prompt surface, `trace_type` now plumbed)*\n`trace_type` selects the trace/telemetry feedback channel around the real run, so arms differ in **feedback richness to the outer optimizer**, not in the score function itself. Runs on the prompt-surface task with `inner_steps=0` (artifact seeding supplies the score gradient; no 5-minute inner-training steps). **Confidence: medium** — the mechanism is real but the magnitude under a nano model may be small; a PARK verdict is a legitimate outcome and is *not* evidence against PR-73 plumbing. **Runtime:** 3 arms × 3 seeds × (4 iters × ~5 calls) ≈ 15–25 min.

In [6]:

from opto.features.recursive_opt import traces
from opto.features.recursive_opt.tracebench import current_task_adapter
TRACES = ["internal", "otel", "hybrid"]
def p2_spec(tt):
    s = prompt_spec(f"o1_tt_{tt}", fixed={"trace_type": tt}); lvl = s["levels"][0]
    lvl["id"] = f"o1_tt_{tt}"; lvl["fixed"]["trace_type"] = tt; return s
for tt in TRACES:
    validate_spec(p2_spec(tt))
adapter = current_task_adapter()
trace_type_plumbed = "trace_type" in getattr(adapter, "PLUMBED_FIELDS", ())
if traces.HAVE_TRACE_IO and trace_type_plumbed:
    p2 = run_variants(p2_spec, TRACES, lambda t: f"o1_tt_{t}")
    if p2:
        save_phase("phase2", p2); decide(p2, "internal")
        disc = {"families": FAMILIES, "budget": BUDGET, "scoring": SCORING,
                "memory_root": MEM_ROOT, "prior_promotion": PROMOTION, "tracebench": TRACEBENCH,
                "levels": [
                  {"id":"o1_disc","surface":"config","family":"optimization_control",
                   "targets":["trace_type"],"constraints":{"trace_type":TRACES},
                   "fixed":{"trainer":P1_WINNER,"optimizer":"OptoPrimeV2"},"iterations":4},
                  {"id":"o2_mix","surface":"family_policy","family":"*",
                   "targets":["trace_type"],"iterations":2}]}
        validate_spec(disc)
        out = run_spec(disc)
        print("discovered mix:", out["results"]["o2_mix"]["artifact"])
        capitalize("decision","*",out["results"]["o2_mix"]["artifact"],
                   out["results"]["o2_mix"]["score"], note="trace-type discovery; confirm before adopting")
else:
    reason = []
    if not traces.HAVE_TRACE_IO:
        reason.append("graph/telemetry backend unavailable")
    if not trace_type_plumbed:
        reason.append("trace_type is not plumbed by the registered adapter")
    payload = {"status": "skipped", "reason": "; ".join(reason), "validated_specs": TRACES}
    save_phase("phase2", payload)
    print("Phase 2 skipped:", payload["reason"])


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 2 agents on 1 inputs:  50%|█████     | 1/2 [00:15<00:15, 15.81s/it]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:15<00:00,  7.90s/it]

Evaluating agent:   0%|          | 0/4 [00:00<?, ?it/s]

...

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:01<00:00,  1.71s/it]

[Step 0] Average test score: 0.0


Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00, 6502.80it/s]

[Step 0] Average test score: -1.0
discovered mix: optimization_control => trace_type=internal
reasoning_control => trace_type=internal
capitalized [decision] *: optimization_control => trace_type=internal
reasoning_contro (score=-0.0029999999999999957)


## Phase 3 — Priors at startup *(prompt surface, namespaced memory)*\n**Self-correction:** reuse previously *hurt* because (a) stale zero-score priors from old runs were visible (append-only memory) and (b) cross-surface artifacts leaked. Fixed by `CAMPAIGN_ID`-namespaced memory + the score-gated promotion + surface-aware seeding. Warm vs cold on the prompt task whose episodes Phase 2 just wrote. **Confidence: medium-high** (the carried prior is a prompt that measurably moved the score in P2's own arms). **Runtime:** 2 arms × 3 seeds × 4 iters ≈ 10–15 min.

In [7]:

def p3_spec(mode):
    return {"families": FAMILIES, "budget": BUDGET, "scoring": SCORING,
        "prior_promotion": PROMOTION, "tracebench": TRACEBENCH,
        "memory_root": MEM_ROOT if mode=="warm" else f"./mem_cold/{CAMPAIGN_ID}",
        "reuse_priors": mode=="warm",
        "levels": [{"id": f"o1_{mode}", "surface": "config", "family": "reasoning_control",
                    "targets": ["starting_artifact","batch_size"],
                      "constraints": {"starting_artifact": ["", "Answer directly.", "Plan step by step, then answer.", "Plan step by step, then verify the answer before replying."]},
                    "fixed": {"trainer": P1_WINNER, "optimizer": "OptoPrimeV2"}, "iterations": 4}]}
p3 = run_variants(p3_spec, ["cold","warm"], lambda m: f"o1_{m}")
if p3:
    save_phase("phase3", p3); decide(p3, "cold")
    if "warm" in p3 and p3["warm"]["mean"] > p3["cold"]["mean"]:
        capitalize("decision","reasoning_control",f"reuse_priors Δ={p3['warm']['mean']-p3['cold']['mean']:+.3f}",
                   p3["warm"]["mean"], note="value of memory at startup", stats=p3.get("warm"))
    else:
        print("memory reuse not capitalized: warm did not beat cold under this bounded run")


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:01<00:00,  1.53s/it]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:01<00:00,  1.53s/it]

...

Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:01<00:00,  1.18s/it]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:01<00:00,  1.18s/it]

[Step 0] Average test score: 0.0


warm = 0.049 ± 0.005 (n=3)
                          warm: Δ=+0.013 (pooled σ=0.006) -> ADOPT
capitalized [decision] reasoning_control: reuse_priors Δ=+0.013 (score=0.04933333333333334)


## Phase 4 — AgenticTrace *(gated: tools need evidence to read)*\n**Self-correction:** tools tied plain because on a flat surface a tie was the only possible outcome, and the memory the tools searched contained no informative failures. P4 now runs only when campaign memory holds failure episodes for the family; otherwise it PARKS with the reason printed. **Runtime when armed:** same envelope as P3.

In [8]:
_mem_gate = MemoryLite(root=MEM_ROOT)
_fails = _mem_gate.similar_failures(family="reasoning", k=1)
if not _fails:
    print("PARKED: no failure episodes in campaign memory for this family yet — "
          "tools would have nothing to read. Re-run after P2/P3 populate memory.")
else:
    def p4_spec(mode):
        s = prompt_spec(f"o1_{mode}"); lvl = s["levels"][0]
        s["budget"] = {**BUDGET, "optimizer_llm_calls": 16}
        if mode=="tools": lvl["agentic"] = True; lvl["tools"] = ["trace_search","note"]
        return s
    p4 = run_variants(p4_spec, ["plain","tools"], lambda m: f"o1_{m}")
    if p4:
        save_phase("phase4", p4); decide(p4, "plain")
        if "tools" in p4 and p4["tools"]["mean"] > p4["plain"]["mean"]:
            capitalize("tool","optimization_control","trace_search", p4["tools"]["mean"],
                       note="tool-evidence improved optimization at equal LLM budget")


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 2 agents on 1 inputs:  50%|█████     | 1/2 [00:15<00:15, 15.68s/it]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:21<00:00,  9.70s/it]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:21<00:00, 10.60s/it]

...

Evaluating agent:  50%|█████     | 2/4 [00:14<00:12,  6.21s/it]

Evaluating agent: 100%|██████████| 4/4 [00:14<00:00,  2.35s/it]

Evaluating agent: 100%|██████████| 4/4 [00:14<00:00,  3.67s/it]

[Step 3] Test/test_score: -0.0006875000000000006
[Step 3] Algo/Average train score: 0.001050000000000001
[Step 3] Update/n_iters: 3
[Step 3] Update/short_term_memory_size: 0
[Step 3] Update/long_term_memory_size: 1
[Step 3] Update/using_short_term_memory: False
[Step 3] Update/using_long_term_memory: True
[Step 3] Update/total_samples: 4
[Step 3] Update/best_candidate_priority: 0.0006875000000000006
[Step 3] Update/best_candidate_mean_score: 0.0006875000000000006
[Step 3] Update/best_candidate_num_rollouts: 4
[Step 3] Update/num_exploration_candidates: 1
[Step 3] Update/exploration_candidates_mean_priority: 0.0006875000000000006
[Step 3] Update/exploration_candidates_mean_score: 0.0006875000000000006
[Step 3] Update/exploration_candidates_average_num_rollouts: 4.0
[Step 3] Sample/mean_score: 0.0025000000000000022
[Step 3] Sample/num_samples: 1
[Step 3] Sample/self.n_epochs: 0
[Step 3] Algo/Number of training samples: 5
[Step 3] Parameter/level_config:21: starting_artifact: 


tools = -0.003 ± 0.006 (n=3)
                         tools: Δ=-0.014 (pooled σ=0.008) -> REJECT


## Phase 5 — Async trainers *(efficiency on the code harness)*\nWall-time to a fixed validator score on the code surface (real work, deterministic eval); quality from Phase 1 remains the guardrail, so the faster thread setting is adopted only when final validator scores tie. Supported trainer paths only. **Confidence: high** for tied-score timing claims. **Runtime:** 2 thread settings × 1 trainer ≈ 5–10 min.

In [4]:
if RUN:
    p5 = {}
    for n_threads in (1, 8):
        t0 = time.time()
        _stats = run_code_variants([P1_WINNER],
            build_fixed=lambda tr: {"trainer": tr, "num_threads": n_threads},
            seeds=(0,), iterations=6)
        p5[f"{P1_WINNER}/threads={n_threads}"] = {"wall_s": time.time()-t0,
            "score": _stats[P1_WINNER]["mean"] if _stats else None}
    save_phase("phase5", p5)
else:
    p5 = load_phase("phase5") or {}
    print("[dry] timing requires key+adapter; previously:", p5 or "no data")
for k,v in (p5 or {}).items(): print(f"  {k:>34}: {v['wall_s']:7.1f}s  score={v['score']}")
if p5: print("ADOPT more threads iff faster at tied score (within 1 sigma of Phase-1).")

PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs (Running sequentially).


Evaluating agent (Running sequentially).
[Step 0] Test/test_score: 0.7999999999999999
[Step 0] Algo/Average train score: 0.8
[Step 0] Update/n_iters: 0
[Step 0] Update/short_term_memory_size: 0
[Step 0] Update/long_term_memory_size: 2
[Step 0] Update/using_short_term_memory: False
[Step 0] Update/using_long_term_memory: True
[Step 0] Update/total_samples: 0
[Step 0] Update/best_candidate_priority: inf
[Step 0] Update/best_candidate_num_rollouts: 0
[Step 0] Update/num_exploration_candidates: 2
[Step 0] Update/exploration_candidates_mean_priority: inf
[Step 0] Update/exploration_candidates_average_num_rollouts: 0.0
[Step 0] Sample/mean_score: 0.8
[Step 0] Sample/num_samples: 2
[Step 0] Sample/self.n_epochs: 0
[Step 0] Algo/Number of training samples: 2
[Step 0] Parameter/__code:6: def _weak_batch_design(self, n, k):
    """Deliberately weak baseline (ignores hard items) — proven improvable to 1.0."""
    return list(range(k))
Epoch: 0. Iteration: 1
Backward (Running sequentially).
Callin

Validating newly proposed candidates: Sampling 2 agents on 1 inputs (Running sequentially).


Sampling training minibatch: Sampling 2 agents on 1 inputs (Running sequentially).


...

Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:00<00:00, 78.36it/s]

Evaluating agent:   0%|          | 0/6 [00:00<?, ?it/s]

Evaluating agent: 100%|██████████| 6/6 [00:00<00:00, 148.56it/s]

[Step 5] Test/test_score: 1.0
[Step 5] Algo/Average train score: 0.9666666666666667
[Step 5] Update/n_iters: 5
[Step 5] Update/short_term_memory_size: 0
[Step 5] Update/long_term_memory_size: 3
[Step 5] Update/using_short_term_memory: False
[Step 5] Update/using_long_term_memory: True
[Step 5] Update/total_samples: 12
[Step 5] Update/best_candidate_priority: 1.0
[Step 5] Update/best_candidate_mean_score: 1.0
[Step 5] Update/best_candidate_num_rollouts: 5
[Step 5] Update/num_exploration_candidates: 2
[Step 5] Update/exploration_candidates_mean_priority: 1.0
[Step 5] Update/exploration_candidates_mean_score: 1.0
[Step 5] Update/exploration_candidates_average_num_rollouts: 5.0
[Step 5] Sample/mean_score: 1.0
[Step 5] Sample/num_samples: 2
[Step 5] Sample/self.n_epochs: 0
[Step 5] Algo/Number of training samples: 12
[Step 5] Parameter/__code:7: def _weak_batch_design(self, n, k):
    """Deliberately weak baseline — improved to prioritize hard/failing indices."""
    # Heuristic: hard/faili

## Phase 6 — skills.md *(distill only from positive signal, validate on the same surface)*\n**Self-correction:** the previous skill was distilled from a 0.000-score memory (its 'best setup' was literally empty) and validated on an incompatible surface. Distillation now requires a positive-score artifact of the SAME family+surface, and the skill is validated as a prompt-surface arm vs the empty control. **Runtime:** 2 arms × 3 seeds ≈ 10 min.

In [ ]:
# Skill phase: distill ONLY from positive signal, validate on the SAME family/surface.
# One SKILL_FAMILY drives gate + distillation + validation arms (live-run leak fixed:
# gate checked "reasoning" while distilling "optimization_control").
SKILL_FAMILY = "reasoning"

def distill_skill(family):
    mem = MemoryLite(root=MEM_ROOT)
    best = mem.best_artifact(family=family)
    fails = mem.similar_failures(family=family, k=3)
    lines = [f"# SKILL - {family}", "", "## Best known setup"]
    if best is not None:                       # robust: callers may probe empty memory
        lines += [f"(score={best.score:.3f})", "```", str(best.content), "```"]
    lines += (["", "## Known failure modes"] +
              [f"- {e.feedback[:140]}" for e in fails] +
              ["", "## Procedure",
               "1. Start from the best known setup above.",
               "2. Verify outputs against the failure modes before accepting a candidate."])
    return "\n".join(lines)

_best = MemoryLite(root=MEM_ROOT).best_artifact(family=SKILL_FAMILY)
if _best is None or _best.score <= 0:
    print(f"PARKED: no positive-score '{SKILL_FAMILY}' artifact in campaign memory — "
          "a skill distilled from zero-signal is noise. Re-run after P2/P3 bank wins.")
else:
    SKILL = distill_skill(SKILL_FAMILY); print(SKILL[:400])
    register_config_values("starting_artifact", [SKILL])   # legal arm for validation
    def p6_spec(mode):
        s = prompt_spec(f"o1_{mode}")           # same family+surface as the skill
        if mode == "skill":
            s["levels"][0]["fixed"]["starting_artifact"] = SKILL
        return s
    p6 = run_variants(p6_spec, ["plain", "skill"], lambda m: f"o1_{m}") or load_phase("phase6") or {}
    if p6:
        save_phase("phase6", p6); decide(p6, "plain")
        if "skill" in p6 and p6["skill"]["mean"] > p6["plain"]["mean"]:
            capitalize("skill", SKILL_FAMILY, SKILL, p6["skill"]["mean"],
                       note="validated SKILL.md (paired vs empty start, same family/surface)",
                       stats=p6["skill"])

## Phase 7 — Terminal-Bench 2 onboarding *(parallel track — never blocks 1–6)*
TB2 is **not** a Trace-Bench family yet: the accessible path is (a) an adapter honoring the exact
`register_task_adapter` contract on 2–3 sandboxed terminal tasks with deterministic checks, then
(b) treating `terminal` as a new family in the Phase-3 transfer spec, seeded by the Phase-6 skill
and the promoted O3 prior. **Capitalization:** everything Phases 1–6 banked (trainer decision,
trace assumption/mix, priors, tools, skill) is the warm start — TB2 begins where the campaign is,
not from zero.

In [ ]:
class TB2AdapterTemplate:
    '''Contract template: implement run_task (and optionally agent_fn) over a local
    terminal harness; scoring must be deterministic checks (cf. make_code_evaluator).'''
    status = "tb2-template (not implemented)"
    def run_task(self, cfg, task_id):
        raise NotImplementedError("wire a sandboxed terminal harness here")
# Seed TB2 with a validated skill only when positive signal exists (empty = bundle default).
TB2_STARTING_ARTIFACT = (
    SKILL if "SKILL" in dir() and isinstance(globals().get("SKILL"), str)
    else distill_skill("optimization_control")
    if (MemoryLite(root=MEM_ROOT).best_artifact(family="optimization_control") or
        type("_", (), {"score": 0})()).score > 0
    else ""
)
if TB2_STARTING_ARTIFACT:
    register_config_values("starting_artifact", [TB2_STARTING_ARTIFACT])
tb2_spec = {"families": {**FAMILIES, "terminal": ["tb2:hello_fs", "tb2:grep_pipeline"]},
    "budget": BUDGET, "scoring": SCORING, "prior_promotion": PROMOTION,
    "memory_root": MEM_ROOT, "reuse_priors": True,
    "levels": [{"id":"o1_tb2","surface":"config","family":"terminal",
                "targets":["starting_artifact","batch_size"],
                "fixed":{"trainer":P1_WINNER,"optimizer":"OptoPrimeV2",
                         "starting_artifact": TB2_STARTING_ARTIFACT},
                "iterations": 4}]}
validate_spec(tb2_spec); print("TB2 transfer spec validated — runnable the day the adapter exists.")

## Campaign decision board

In [12]:

board = {}
for ph in ["phase0_gates","phase1","phase2","phase3","phase4","phase5","phase6"]:
    d = load_phase(ph)
    board[ph] = "no data" if d is None else (d if ph=="phase0_gates" else
        {k:(f"{v['mean']:+.3f}±{v['std']:.3f}" if isinstance(v,dict) and "mean" in v else v)
         for k,v in d.items()})
print(json.dumps(board, indent=1))
mem = MemoryLite(root=MEM_ROOT)
print("\ncapitalized assets (all memory):", mem.summary())
print("current-run capitalized assets:")
for kind in ("decision","assumption","tool","skill"):
    for a in mem.artifact_history(kind=kind):
        if getattr(a, "ts", 0.0) < RUN_STARTED:
            continue
        print(f"  [{kind}] {a.family}: {str(a.content)[:70]} (score={a.score:.3f})")


{
 "phase0_gates": {
  "tests": "70 passed in 2.12s",
  "adapter": "REAL (Trace-Bench bundle adapter; tasks_root=/home/xav/code/Trace-Bench/benchmarks/LLM4AD/benchmark_tasks; max_examples=1; inner_steps=1; inner_candidates=1; allowed_inner_trainers=['MinibatchAlgorithm', 'PrioritySearch'])",
  "trace_io": "AVAILABLE"
 },
 "phase1": {
  "MinibatchAlgorithm": "+1.000\u00b10.000",
  "PrioritySearch": "+1.000\u00b10.000"
 },
 "phase2": {
  "internal": "+0.014\u00b10.026",
  "otel": "+0.014\u00b10.021",
  "hybrid": "+0.017\u00b10.007"
 },
 "phase3": {
  "cold": "+0.036\u00b10.007",
  "warm": "+0.049\u00b10.005"
 },
 "phase4": {
  "plain": "+0.012\u00b10.011",
  "tools": "-0.003\u00b10.006"
 },
 "phase5": {
  "MinibatchAlgorithm/threads=1": {
   "wall_s": 11.665836572647095,
   "score": 1.0
  },
  "MinibatchAlgorithm/threads=8": {
   "wall_s": 0.01799798011779785,
   "score": 0.8
  }
 },
 "phase6": {
  "plain": "+0.011\u00b10.005",
  "skill": "+0.040\u00b10.011"
 }
}

capitalized assets (all

## Confirmation round (larger-eval stress test)\n**Kernel notes for the running session:** re-running **P1/P5** requires a kernel **restart** because they depend on library code loaded into the kernel. The confirm cells below only need cell 1 re-run in the current kernel. They re-test P2/P3/P6 at `max_examples=8` with n=3 seeds. Treat any `-1e9` sentinel, `-inf`, or failed `phaseN_confirm` write as confirmation failure/PARK before interpreting means; these cells are robustness stress tests, not guaranteed cleaner estimates. P6 can also fail if the distilled skill plus memory makes the live request too large, in which case the skill should be compacted before confirmation.

In [3]:
import copy as _copy
def confirm(make_spec, variants, level_id, phase_name, max_examples=8):
    """DRY confirm runner: same paired spec, bigger eval set, fresh phase file."""
    def _spec(v):
        s = _copy.deepcopy(make_spec(v))
        s["tracebench"] = {**s.get("tracebench", {}), "max_examples": max_examples, "inner_steps": 0}
        return s
    out = run_variants(_spec, variants, level_id) or load_phase(phase_name) or {}
    if out:
        save_phase(phase_name, out); decide(out, str(variants[0]))
    return out

# P2c — confirm the discovered tracing margin (hybrid vs internal only).
if (load_phase("phase2") or {}):
    p2c = confirm(lambda tt: prompt_spec(f"o1_ttc_{tt}", fixed={"trace_type": tt}),
                  ["internal", "hybrid"], lambda t: f"o1_ttc_{t}", "phase2_confirm")
    if p2c and "hybrid" in p2c and p2c["hybrid"]["mean"] > p2c["internal"]["mean"]:
        capitalize("decision", "reasoning", "trace_type=hybrid (confirmed paired)",
                   p2c["hybrid"]["mean"], stats=p2c["hybrid"])
else:
    print("phase2 has no data yet — run Phase 2 first.")

PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

/home/xav/miniconda3/envs/humanllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sampling training minibatch: Sampling 2 agents on 1 inputs:  50%|█████     | 1/2 [00:30<00:30, 30.63s/it]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:30<00:00, 12.77s/it]

...

Evaluating agent:  50%|█████     | 2/4 [00:13<00:11,  5.85s/it]

Evaluating agent:  75%|███████▌  | 3/4 [00:14<00:03,  3.26s/it]

Evaluating agent: 100%|██████████| 4/4 [00:14<00:00,  2.29s/it]

Evaluating agent: 100%|██████████| 4/4 [00:14<00:00,  3.74s/it]

[Step 3] Test/test_score: 0.0026874999999999954
[Step 3] Algo/Average train score: -124999999.99996875
[Step 3] Update/n_iters: 3
[Step 3] Update/short_term_memory_size: 0
[Step 3] Update/long_term_memory_size: 7
[Step 3] Update/using_short_term_memory: False
[Step 3] Update/using_long_term_memory: True
[Step 3] Update/total_samples: 12
[Step 3] Update/best_candidate_priority: 0.004812499999999997
[Step 3] Update/best_candidate_mean_score: 0.004812499999999997
[Step 3] Update/best_candidate_num_rollouts: 2
[Step 3] Update/num_exploration_candidates: 2
[Step 3] Update/exploration_candidates_mean_priority: 0.0019999999999999914
[Step 3] Update/exploration_candidates_mean_score: 0.0019999999999999914
[Step 3] Update/exploration_candidates_average_num_rollouts: 3.0
[Step 3] Sample/mean_score: -0.0003125000000000211
[Step 3] Sample/num_samples: 2
[Step 3] Sample/self.n_epochs: 0
[Step 3] Algo/Number of training samples: 8
[Step 3] Parameter/level_config:5: starting_artifact: 
hybrid = -6666

In [4]:
# P3 confirm — warm vs cold at max_examples=8 (was 1-example evals).
if (load_phase("phase3") or {}):
    p3c = confirm(p3_spec, ["cold", "warm"], lambda m: f"o1_{m}", "phase3_confirm")
    if p3c and "warm" in p3c and p3c["warm"]["mean"] > p3c["cold"]["mean"]:
        capitalize("decision", "reasoning_control",
                   f"reuse_priors confirmed Delta={p3c['warm']['mean']-p3c['cold']['mean']:+.3f}@8ex",
                   p3c["warm"]["mean"], stats=p3c["warm"])
else:
    print("phase3 has no data yet — run Phase 3 first.")

PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 2 agents on 1 inputs:  50%|█████     | 1/2 [00:05<00:05,  5.05s/it]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:05<00:00,  2.56s/it]

Evaluating agent:   0%|          | 0/4 [00:00<?, ?it/s]

...

Evaluating agent:   0%|          | 0/4 [00:00<?, ?it/s]

Evaluating agent:  25%|██▌       | 1/4 [00:01<00:05,  1.93s/it]

Evaluating agent:  50%|█████     | 2/4 [00:02<00:01,  1.09it/s]

Evaluating agent: 100%|██████████| 4/4 [00:02<00:00,  1.80it/s]

[Step 3] Test/test_score: 0.06025
[Step 3] Algo/Average train score: -124999999.948625
[Step 3] Update/n_iters: 3
[Step 3] Update/short_term_memory_size: 0
[Step 3] Update/long_term_memory_size: 7
[Step 3] Update/using_short_term_memory: False
[Step 3] Update/using_long_term_memory: True
[Step 3] Update/total_samples: 12
[Step 3] Update/best_candidate_priority: 0.061
[Step 3] Update/best_candidate_mean_score: 0.061
[Step 3] Update/best_candidate_num_rollouts: 2
[Step 3] Update/num_exploration_candidates: 2
[Step 3] Update/exploration_candidates_mean_priority: 0.05925
[Step 3] Update/exploration_candidates_mean_score: 0.05925
[Step 3] Update/exploration_candidates_average_num_rollouts: 3.0
[Step 3] Sample/mean_score: 0.06
[Step 3] Sample/num_samples: 2
[Step 3] Sample/self.n_epochs: 0
[Step 3] Algo/Number of training samples: 8
[Step 3] Parameter/level_config:11: starting_artifact: Answer directly.
batch_size: 4
warm = -333333333.320 ± 471404520.800 (n=3)
                          warm:

In [5]:
# P6 confirm — skill vs plain at max_examples=8.
if "p6_spec" in dir():
    p6c = confirm(p6_spec, ["plain", "skill"], lambda m: f"o1_{m}", "phase6_confirm")
    if p6c and "skill" in p6c and p6c["skill"]["mean"] > p6c["plain"]["mean"]:
        capitalize("skill", SKILL_FAMILY, SKILL, p6c["skill"]["mean"],
                   note="skill confirmed at 8 examples", stats=p6c["skill"])
else:
    print("P6 was PARKED or not yet run — no skill to confirm.")